In [ ]:
# Task
# Design and implement a multi-agent workflow using LangGraph (or similar framework).

#1. Agent Design
# Define at least 3 agents, such as:

# Retrieval Agent
# Reasoning/Answer Agent
# Validation Agent
# Explain briefly (in comments or code):

# Each agent’s role
# Input/output
# 2. Graph Workflow Implementation
# Write code or pseudo-code to:

# Define state
# Add nodes (agents)
# Define edges
# Implement conditional logic
#  Must include:

# Retry loop if validation fails
# Clear start and end states
#  3. State Management
# Show how state evolves across steps:

# Query
# Context
# Intermediate reasoning
# Final answer
# Validation flag
#  4. Task Delegation & Communication
# Demonstrate:

# How agents pass information
# How decisions are made between agents
#  Expected Outcome
# A clear multi-step, graph-based agent system that:

# Handles complex queries
# Demonstrates reasoning + validation
# Uses proper orchestration 

In [12]:
# State Definition

In [13]:
from typing import TypedDict

class SupportState(TypedDict):
    query: str
    query_type: str
    context: str
    reasoning: str
    answer: str
    validation: bool
    feedback: str
    retry_count: int

In [14]:
# Agent Implementations
# Retrieval Agent
knowledge_base = {
    "transaction": "If money is deducted but transaction failed, amount is usually auto-refunded within 3-5 business days.",
    "fraud": "If you detect fraud, immediately block card and report to bank support.",
    "refund": "Refunds are processed within 5-7 working days depending on merchant.",
    "faq": "You can check account info, reset password, and update KYC from dashboard."
}

def retrieval_agent(state):

    query = state["query"].lower()

    if "fraud" in query:
        query_type = "fraud"
    elif "refund" in query:
        query_type = "refund"
    elif "transaction" in query:
        query_type = "transaction"
    else:
        query_type = "faq"

    context = knowledge_base[query_type]

    return {
        "query_type": query_type,
        "context": context
    }

In [15]:
# Reasoning Agent

# Uses LLM (NVIDIA API optional)

from langchain_nvidia_ai_endpoints import ChatNVIDIA

llm = ChatNVIDIA(model="meta/llama3-70b-instruct")

def reasoning_agent(state):

    prompt = f"""
    Customer Query:
    {state['query']}

    Context:
    {state['context']}

    Provide step-by-step reasoning and final answer.
    """

    response = llm.invoke(prompt)

    return {
        "reasoning": response.content,
        "answer": response.content
    }

In [16]:
# Validation Agent
def validation_agent(state):

    answer = state["answer"]
    context = state["context"]

    validation_prompt = f"""
    Check if answer is correct based on context.

    Context:
    {context}

    Answer:
    {answer}

    Return:
    VALID or INVALID
    """

    result = llm.invoke(validation_prompt)

    is_valid = "VALID" in result.content

    return {
        "validation": is_valid,
        "feedback": result.content
    }

In [17]:
# Retry Logic Agent
def retry_agent(state):

    retry_count = state["retry_count"] + 1

    improved_prompt = f"""
    Improve answer using feedback.

    Feedback:
    {state['feedback']}

    Previous Answer:
    {state['answer']}
    """

    response = llm.invoke(improved_prompt)

    return {
        "answer": response.content,
        "retry_count": retry_count
    }

In [18]:
from langgraph.graph import StateGraph, END

workflow = StateGraph(SupportState)

# nodes
workflow.add_node("retrieval", retrieval_agent)
workflow.add_node("reasoning", reasoning_agent)
workflow.add_node("validation", validation_agent)
workflow.add_node("retry", retry_agent)

# edges
workflow.set_entry_point("retrieval")

workflow.add_edge("retrieval", "reasoning")

workflow.add_edge("reasoning", "validation")

# conditional retry loop
def check_validation(state):

    if state["validation"] == True:
        return END

    elif state["retry_count"] < 2:
        return "retry"

    else:
        return END


workflow.add_conditional_edges(
    "validation",
    check_validation
)

workflow.add_edge("retry", "validation")

graph = workflow.compile()

In [19]:
state = {
    "query": "My transaction failed but money deducted",
    "query_type": "",
    "context": "",
    "reasoning": "",
    "answer": "",
    "validation": False,
    "feedback": "",
    "retry_count": 0
}

In [ ]:
result = graph.invoke(state)
print(result["answer"])

Here's the step-by-step reasoning and final answer:

**Step 1: Understand the issue**
The customer's transaction has failed, but the money has been deducted from their account.

**Step 2: Identify the possible cause**
This issue could be due to a technical glitch or a payment processing error.

**Step 3: Check the refund policy**
According to our refund policy, if the money is deducted but the transaction fails, the amount is usually auto-refunded within 3-5 business days.

**Step 4: Provide a solution**
In this case, we can assure the customer that the amount will be auto-refunded within 3-5 business days. If the refund doesn't happen within the specified timeframe, the customer can reach out to our support team for further assistance.

**Final Answer:**
"Dear customer, we apologize for the inconvenience caused by the failed transaction. Please be assured that the deducted amount will be auto-refunded to your account within 3-5 business days. If you don't receive the refund within thi